In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv("StudentPerformanceFactors.csv")

print("Shape:", df.shape)

display(df.head())

df.info()

display(df.describe())

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().copy()

In [ ]:
numerical_features = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions"
]

categorical_features = [
    "Teacher_Quality",
    "Parental_Education_Level"
]

X = df[numerical_features + categorical_features]

y = df["Exam_Score"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [ ]:
numeric_imputer = SimpleImputer(strategy="median")

X_train_num = numeric_imputer.fit_transform(
    X_train[numerical_features]
)

X_test_num = numeric_imputer.transform(
    X_test[numerical_features]
)

In [ ]:
categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train_cat_imputed = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

X_test_cat_imputed = categorical_imputer.transform(
    X_test[categorical_features]
)

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(
    X_train_cat_imputed
)

X_test_cat = encoder.transform(
    X_test_cat_imputed
)

print(encoder.get_feature_names_out(categorical_features))

In [ ]:
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(
    X_train_num
)

X_test_num_scaled = scaler.transform(
    X_test_num
)

In [ ]:
X_train_processed = np.concatenate(
    [X_train_num_scaled, X_train_cat],
    axis=1
)

X_test_processed = np.concatenate(
    [X_test_num_scaled, X_test_cat],
    axis=1
)

In [ ]:
feature_names = numerical_features + list(
    encoder.get_feature_names_out(categorical_features)
)

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed.head())

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

In [ ]:
model = LinearRegression()

model.fit(
    X_train_processed,
    y_train
)

y_pred = model.predict(X_test_processed)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f}")
print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")